# Landing zone: offline NYC 311 snapshots

For learning the first warehouse layer after setting up the repository's `.venv`. You will load fictional observations, inspect batch audit records, trace one request over time, and replay without duplicates. No API token or network is needed.

1. Locate the project and database.
2. Load three fixed synthetic batches.
3. Inspect the audit and request history.
4. Replay and verify unchanged audit records.

The default is the existing `data/nyc_open_data.db`. Only the `landing` schema is created; the exploratory table is preserved. Close other DuckDB connections first. Select the `.venv` Python kernel.

In [ ]:
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "ingestion/landing.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or notebook directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ingestion.landing import DEFAULT_DATABASE_PATH, load_synthetic

# Optional override for a disposable demo database; otherwise use the project database.
DATABASE_PATH = Path(os.getenv("NYC311_DEMO_DATABASE", str(DEFAULT_DATABASE_PATH)))
print(f"Landing database: {DATABASE_PATH}")


## Load three daily observations

Each batch contains three fictional requests. DEMO-001 reopens, DEMO-002 changes agency, and DEMO-003 stays unchanged. Source dates are strings; the fixed February 2026 capture times are timezone aware. Actual ingestion start/end times are recorded separately. Running this cell again safely replays identical batches.

In [ ]:
results = load_synthetic(DATABASE_PATH)
display(pd.DataFrame([
    {"batch_id": result.batch_id, "accepted_rows": result.accepted_rows,
     "replayed": result.replayed}
    for result in results
]))
batch_ids = [result.batch_id for result in results]
placeholders = ",".join("?" for _ in batch_ids)


## Inspect the ingestion audit

Expect three successful demo batches with three committed rows each, zero rejected rows, and zero API pages. `accepted_rows` counts committed rows. `rejected_rows` counts structurally invalid records; a transaction failure can have zero rejected records and zero accepted rows.

Only successful batches should feed downstream models. A failed or interrupted batch ID cannot be reused; retry with a new ID. Metadata validation errors such as a missing capture timezone are rejected before creating an audit row.

In [ ]:
audit_query = f"""
SELECT batch_id, mode, status, snapshot_at_utc, started_at_utc, finished_at_utc,
       page_count, fetched_rows, accepted_rows, rejected_rows
FROM landing.ingestion_batch
WHERE batch_id IN ({placeholders})
ORDER BY batch_id
"""
with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
    audit_before = con.execute(audit_query, batch_ids).fetchall()
    display(con.execute(audit_query, batch_ids).df())


## Trace a request across snapshots

All nine demo observations remain in landing, including repeated values. The first request shows Open → Closed → Open. These are observed states; capture time does not tell us the exact source transition time. Source `due_date` means expected agency update time, not a universal closure SLA.

In [ ]:
with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
    history = con.execute(f"""
        SELECT r.unique_key, r.snapshot_at_utc, r.agency, r.status,
               r.closed_date, r.incident_zip, r.record_source
        FROM landing.request_snapshots AS r
        JOIN landing.ingestion_batch AS b ON r.batch_id = b.batch_id
        WHERE b.status = 'success' AND r.batch_id IN ({placeholders})
        ORDER BY r.unique_key, r.snapshot_at_utc
    """, batch_ids).df()
display(history)
assert len(history) == 9


## Replay without changing evidence

The same batch IDs, observations and metadata produce a no-op. Input row order does not affect the checksum. Reusing a batch ID with changed content or metadata raises `BatchConflictError`; it never overwrites successful observations.

In [ ]:
replayed = load_synthetic(DATABASE_PATH)
assert all(result.replayed for result in replayed)
with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
    audit_after = con.execute(audit_query, batch_ids).fetchall()
    count_after = con.execute(f"""
        SELECT count(*) FROM landing.request_snapshots
        WHERE batch_id IN ({placeholders})
    """, batch_ids).fetchone()[0]
assert audit_after == audit_before
assert count_after == 9
print("Replay verified: nine observations; original audit timestamps unchanged.")


## Exercise: explain the unchanged request

How many rows should DEMO-003 have in landing, and how many distinct status values? Fill in your prediction before running the answer below. Think about why unchanged observations still matter.

The next layer will prepare typed values and eventually build Vault changes. This notebook does not create staging or Vault models. Keep database files local and clear notebook outputs before committing.

In [ ]:
# Answer: three observations, one distinct status. Each batch preserves evidence.
with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
    display(con.execute(f"""
        SELECT unique_key, count(*) AS observations,
               count(DISTINCT status) AS distinct_statuses
        FROM landing.request_snapshots
        WHERE unique_key = 'DEMO-003' AND batch_id IN ({placeholders})
        GROUP BY unique_key
    """, batch_ids).df())
